<a href="https://colab.research.google.com/github/alexandrelombard/ai54-notebooks/blob/master/05_seq2seq_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import random
import time
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd

import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)


True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
# Load dataset
df = pd.read_csv('english-french.csv')

# Optionally subsample for quick demo training
MAX_PAIRS = 50000  # try 20_000 to 100_000 for better quality if you have time/GPU
if MAX_PAIRS is not None:
    df = df.iloc[:MAX_PAIRS]

print("Dataset size:", len(df))
print(df.head())


Dataset size: 50000
  english      french
0     Hi.      Salut!
1    Run!     Cours !
2    Run!    Courez !
3    Who?       Qui ?
4    Wow!  Ça alors !


In [ ]:
# Tokenization helpers
SOS = '<sos>'
EOS = '<eos>'
PAD = '<pad>'
UNK = '<unk>'

SPECIAL_TOKENS = [PAD, SOS, EOS, UNK]
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

def tokenize(text: str):
    return word_tokenize(str(text).strip().lower())

class Vocab:
    def __init__(self, max_size: int = 20000, min_freq: int = 1):
        self.stoi: Dict[str, int] = {}
        self.itos: List[str] = []
        self.max_size = max_size
        self.min_freq = min_freq

        # init with specials in fixed order to match indices defined above
        for tok in SPECIAL_TOKENS:
            self.add_token(tok)

    def add_token(self, tok: str):
        if tok not in self.stoi:
            self.stoi[tok] = len(self.itos)
            self.itos.append(tok)

    def build(self, texts: List[List[str]]):
        from collections import Counter
        counter = Counter()
        for toks in texts:
            counter.update(toks)
        # Exclude specials from frequency sorting
        frequent = [(t, c) for t, c in counter.items() if c >= self.min_freq and t not in SPECIAL_TOKENS]
        frequent.sort(key=lambda x: (-x[1], x[0]))
        for t, _ in frequent[: max(0, self.max_size - len(self.itos))]:
            self.add_token(t)

    def __len__(self):
        return len(self.itos)

    def encode(self, toks: List[str]) -> List[int]:
        return [self.stoi.get(t, UNK_IDX) for t in toks]

    def decode(self, ids: List[int]) -> List[str]:
        return [self.itos[i] if 0 <= i < len(self.itos) else UNK for i in ids]



In [ ]:
# Build vocabularies
src_texts_tok = [tokenize(t) for t in df['english'].tolist()]
tgt_texts_tok = [tokenize(t) for t in df['french'].tolist()]

SRC_VOCAB_SIZE = 20000
TGT_VOCAB_SIZE = 25000

src_vocab = Vocab(max_size=SRC_VOCAB_SIZE)
src_vocab.build(src_texts_tok)

tgt_vocab = Vocab(max_size=TGT_VOCAB_SIZE)
tgt_vocab.build([[SOS] + toks + [EOS] for toks in tgt_texts_tok])

print("Vocab sizes:", len(src_vocab), len(tgt_vocab))


Vocab sizes: 5937 12563


In [ ]:
# Dataset and collation
class TranslationDataset(Dataset):
    def __init__(self, src_tok: List[List[str]], tgt_tok: List[List[str]]):
        assert len(src_tok) == len(tgt_tok)
        self.src_tok = src_tok
        self.tgt_tok = tgt_tok

    def __len__(self):
        return len(self.src_tok)

    def __getitem__(self, idx):
        src = self.src_tok[idx]
        tgt = self.tgt_tok[idx]
        # For target, add SOS/EOS
        tgt_full = [SOS] + tgt + [EOS]
        return src, tgt_full


def collate_fn(batch: List[Tuple[List[str], List[str]]]):
    # Convert tokens to ids and pad
    src_ids = [src_vocab.encode(s) for s, _ in batch]
    tgt_ids = [tgt_vocab.encode(t) for _, t in batch]

    # Prepare input and output for decoder: input is without last token, output is without first token
    tgt_in = [t[:-1] for t in tgt_ids]
    tgt_out = [t[1:] for t in tgt_ids]

    def pad_to_max(seqs: List[List[int]], pad_idx: int) -> Tuple[torch.Tensor, List[int]]:
        max_len = max(len(s) for s in seqs)
        padded = [s + [pad_idx] * (max_len - len(s)) for s in seqs]
        lengths = [len(s) for s in seqs]
        return torch.tensor(padded, dtype=torch.long), lengths

    src_batch, src_lens = pad_to_max(src_ids, PAD_IDX)
    tgt_in_batch, tgt_in_lens = pad_to_max(tgt_in, PAD_IDX)
    tgt_out_batch, _ = pad_to_max(tgt_out, PAD_IDX)

    # Create padding masks (True for pads for PyTorch transformer)
    src_pad_mask = (src_batch == PAD_IDX)
    tgt_pad_mask = (tgt_in_batch == PAD_IDX)

    return src_batch.to(device), tgt_in_batch.to(device), tgt_out_batch.to(device), src_pad_mask.to(device), tgt_pad_mask.to(device)


# Train/val split
VAL_RATIO = 0.01
n_total = len(src_texts_tok)
n_val = max(1, int(n_total * VAL_RATIO))
val_src = src_texts_tok[:n_val]
val_tgt = tgt_texts_tok[:n_val]
train_src = src_texts_tok[n_val:]
train_tgt = tgt_texts_tok[n_val:]

train_ds = TranslationDataset(train_src, train_tgt)
val_ds = TranslationDataset(val_src, val_tgt)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


Train batches: 387, Val batches: 4


In [ ]:
# Positional encoding and Transformer Seq2Seq model
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, : x.size(1), :]


class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size: int, tgt_vocab_size: int, d_model: int = 256, nhead: int = 8,
                 num_encoder_layers: int = 3, num_decoder_layers: int = 3, dim_feedforward: int = 512,
                 dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(d_model)
        self.pos_decoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead,
                                          num_encoder_layers=num_encoder_layers,
                                          num_decoder_layers=num_decoder_layers,
                                          dim_feedforward=dim_feedforward,
                                          dropout=dropout, batch_first=True)
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src_tokens: torch.Tensor, tgt_tokens_in: torch.Tensor,
                src_key_padding_mask: torch.Tensor, tgt_key_padding_mask: torch.Tensor) -> torch.Tensor:
        # src_tokens: (B, S), tgt_tokens_in: (B, T)
        src_emb = self.pos_encoder(self.src_embed(src_tokens) * math.sqrt(self.d_model))  # (B, S, D)
        tgt_emb = self.pos_decoder(self.tgt_embed(tgt_tokens_in) * math.sqrt(self.d_model))  # (B, T, D)

        # Generate causal mask for target to prevent attending to future tokens
        T = tgt_tokens_in.size(1)
        causal_mask = torch.triu(torch.full((T, T), float('-inf'), device=tgt_tokens_in.device), diagonal=1)

        out = self.transformer(src=src_emb,
                               tgt=tgt_emb,
                               tgt_mask=causal_mask,
                               src_key_padding_mask=src_key_padding_mask,
                               tgt_key_padding_mask=tgt_key_padding_mask,
                               memory_key_padding_mask=src_key_padding_mask)
        logits = self.generator(out)  # (B, T, Vtgt)
        return logits



In [ ]:
# Instantiate model, optimizer, loss
D_MODEL = 256
N_HEAD = 8
ENC_LAYERS = 3
DEC_LAYERS = 3
FF_DIM = 512
DROP = 0.1

model = TransformerSeq2Seq(len(src_vocab), len(tgt_vocab), d_model=D_MODEL, nhead=N_HEAD,
                           num_encoder_layers=ENC_LAYERS, num_decoder_layers=DEC_LAYERS,
                           dim_feedforward=FF_DIM, dropout=DROP).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


In [ ]:
# Training and evaluation loops
def train_epoch(model: nn.Module, loader: DataLoader) -> float:
    model.train()
    total_loss = 0.0
    for i, (src, tgt_in, tgt_out, src_pad_mask, tgt_pad_mask) in enumerate(loader):
        optimizer.zero_grad(set_to_none=True)
        logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask)  # (B, T, V)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B * T, V), tgt_out.reshape(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        if (i + 1) % 100 == 0:
            print(f"  batch {i+1}/{len(loader)} - loss: {total_loss/(i+1):.4f}")
    return total_loss / max(1, len(loader))


def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for src, tgt_in, tgt_out, src_pad_mask, tgt_pad_mask in loader:
            logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V), tgt_out.reshape(B * T))
            total_loss += loss.item()
    return total_loss / max(1, len(loader))



In [ ]:
EPOCHS = 10  # increase for better results
print_every_epoch = True

best_val = float('inf')
start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader)
    val_loss = evaluate(model, val_loader)
    dt = time.time() - t0
    if print_every_epoch:
        print(f"Epoch {epoch}/{EPOCHS} - train loss: {train_loss:.4f} - val loss: {val_loss:.4f} - time: {dt:.1f}s")
    if val_loss < best_val:
        best_val = val_loss
        # lightweight checkpoint in memory; optionally save to disk
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

print(f"Training done in {time.time()-start_time:.1f}s. Best val loss: {best_val:.4f}")


C:\Users\alombard.PCP-GI-05\miniconda3\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


  batch 100/387 - loss: 5.3045
  batch 200/387 - loss: 4.6140
  batch 300/387 - loss: 4.2477


C:\Users\alombard.PCP-GI-05\miniconda3\Lib\site-packages\torch\nn\modules\transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Epoch 1/10 - train loss: 4.0260 - val loss: 3.1971 - time: 11.5s
  batch 100/387 - loss: 2.9712
  batch 200/387 - loss: 2.8891
  batch 300/387 - loss: 2.8220
Epoch 2/10 - train loss: 2.7655 - val loss: 2.8124 - time: 11.2s
  batch 100/387 - loss: 2.3895
  batch 200/387 - loss: 2.3491
  batch 300/387 - loss: 2.3110
Epoch 3/10 - train loss: 2.2823 - val loss: 2.6008 - time: 11.3s
  batch 100/387 - loss: 1.9970
  batch 200/387 - loss: 1.9826
  batch 300/387 - loss: 1.9638
Epoch 4/10 - train loss: 1.9457 - val loss: 2.5004 - time: 11.5s
  batch 100/387 - loss: 1.7142
  batch 200/387 - loss: 1.7054
  batch 300/387 - loss: 1.7026
Epoch 5/10 - train loss: 1.6963 - val loss: 2.3499 - time: 11.4s
  batch 100/387 - loss: 1.5066
  batch 200/387 - loss: 1.5066
  batch 300/387 - loss: 1.5012
Epoch 6/10 - train loss: 1.4986 - val loss: 2.2451 - time: 11.3s
  batch 100/387 - loss: 1.3257
  batch 200/387 - loss: 1.3310
  batch 300/387 - loss: 1.3382
Epoch 7/10 - train loss: 1.3401 - val loss: 2.1647 -

In [ ]:
# Greedy decoder for translation (everytime we select only the most probable token)
@torch.no_grad()
def greedy_decode(model: nn.Module, src_sentence: str, max_len: int = 40) -> str:
    model.eval()
    src_toks = tokenize(src_sentence)
    src_ids = torch.tensor([src_vocab.encode(src_toks)], dtype=torch.long, device=device)
    src_pad_mask = (src_ids == PAD_IDX)

    # Start with SOS
    generated = [SOS_IDX]

    for _ in range(max_len):
        tgt_in = torch.tensor([generated], dtype=torch.long, device=device)
        tgt_pad_mask = (tgt_in == PAD_IDX)
        logits = model(src_ids, tgt_in, src_pad_mask, tgt_pad_mask)  # (1, T, V)
        next_token = int(logits[0, -1].argmax(dim=-1).item())
        if next_token == EOS_IDX:
            break
        generated.append(next_token)
    # Convert to tokens, drop SOS
    toks = tgt_vocab.decode(generated[1:])
    # Stop at EOS if present
    if EOS in toks:
        toks = toks[: toks.index(EOS)]
    return ' '.join(toks)

In [ ]:
# Top-k decoder for translation (everytime we select the top-k probable token)
@torch.no_grad()
def topk_decode(model: nn.Module, src_sentence: str, k: int = 2, temperature = 1.0, max_len: int = 40) -> str:
    model.eval()
    src_toks = tokenize(src_sentence)
    src_ids = torch.tensor([src_vocab.encode(src_toks)], dtype=torch.long, device=device)
    src_pad_mask = (src_ids == PAD_IDX)

    # Start with SOS
    generated = [SOS_IDX]

    for _ in range(max_len):
        tgt_in = torch.tensor([generated], dtype=torch.long, device=device)
        tgt_pad_mask = (tgt_in == PAD_IDX)
        logits = model(src_ids, tgt_in, src_pad_mask, tgt_pad_mask)  # (1, T, V)

        topk = torch.topk(logits[0, -1], k=k)
        topk_probabilities = torch.nn.functional.softmax(topk.values / temperature)
        next_token = topk.indices[torch.multinomial(topk_probabilities, 1)]

        if next_token == EOS_IDX:
            break
        generated.append(next_token)
    # Convert to tokens, drop SOS
    toks = tgt_vocab.decode(generated[1:])
    # Stop at EOS if present
    if EOS in toks:
        toks = toks[: toks.index(EOS)]
    return ' '.join(toks)

In [ ]:
# Top-p (nucleus) decoder for translation
@torch.no_grad()
def topp_decode(model: nn.Module, src_sentence: str, p: float = 0.5, temperature: float = 1.0, max_len: int = 40) -> str:
    model.eval()
    src_toks = tokenize(src_sentence)
    src_ids = torch.tensor([src_vocab.encode(src_toks)], dtype=torch.long, device=device)
    src_pad_mask = (src_ids == PAD_IDX)

    # Start with SOS
    generated = [SOS_IDX]

    for _ in range(max_len):
        tgt_in = torch.tensor([generated], dtype=torch.long, device=device)
        tgt_pad_mask = (tgt_in == PAD_IDX)
        logits = model(src_ids, tgt_in, src_pad_mask, tgt_pad_mask)  # (1, T, V)

        # Take last-step logits and apply temperature
        step_logits = logits[0, -1] / max(temperature, 1e-6)
        probs = torch.nn.functional.softmax(step_logits, dim=-1)

        # Sort probabilities and compute cumulative sum
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)

        # Determine the selected set (at least one token)
        cutoff_idx = int((cumulative > p).nonzero(as_tuple=False)[:1].flatten().item()) if (cumulative > p).any() else (sorted_probs.numel() - 1)
        keep_up_to = max(cutoff_idx, 0)
        selected_probs = sorted_probs[: keep_up_to + 1]
        selected_indices = sorted_indices[: keep_up_to + 1]

        # Renormalize and sample
        selected_probs = selected_probs / selected_probs.sum()
        sampled_pos = torch.multinomial(selected_probs, 1).item()
        next_token = int(selected_indices[sampled_pos].item())

        if next_token == EOS_IDX:
            break
        generated.append(next_token)
    # Convert to tokens, drop SOS
    toks = tgt_vocab.decode(generated[1:])
    # Stop at EOS if present
    if EOS in toks:
        toks = toks[: toks.index(EOS)]
    return ' '.join(toks)

In [ ]:
# Beam search decoder for translation
@torch.no_grad()
def beam_search_decode(model: nn.Module, src_sentence: str, beam_width: int = 2, temperature: float = 1.0, max_len: int = 40) -> str:
    model.eval()
    src_toks = tokenize(src_sentence)
    src_ids = torch.tensor([src_vocab.encode(src_toks)], dtype=torch.long, device=device)
    src_pad_mask = (src_ids == PAD_IDX)

    # Each beam is a tuple: (token_ids_list, cumulative_log_prob, ended)
    beams = [([SOS_IDX], 0.0, False)]
    finished = []

    for _ in range(max_len):
        new_beams = []
        all_ended = True
        for tokens, logp, ended in beams:
            if ended:
                new_beams.append((tokens, logp, True))
                continue
            all_ended = False
            tgt_in = torch.tensor([tokens], dtype=torch.long, device=device)
            tgt_pad_mask = (tgt_in == PAD_IDX)
            logits = model(src_ids, tgt_in, src_pad_mask, tgt_pad_mask)  # (1, T, V)
            step_logits = logits[0, -1] / max(temperature, 1e-6)
            # Using log_probs for numerical stability (sum instead of product)
            log_probs = torch.nn.functional.log_softmax(step_logits, dim=-1)
            topk_log_probs, topk_indices = torch.topk(log_probs, k=beam_width)
            for k in range(topk_indices.size(0)):
                next_tok = int(topk_indices[k].item())
                new_tokens = tokens + [next_tok]
                new_logp = logp + float(topk_log_probs[k].item())
                ended_new = (next_tok == EOS_IDX)
                if ended_new:
                    finished.append((new_tokens, new_logp, True))
                new_beams.append((new_tokens, new_logp, ended_new))
        if all_ended:
            break
        # Keep top-k beams
        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_width]

    # Select the best finished beam if available; otherwise, best current beam
    candidates = finished if finished else beams
    best_tokens, _, _ = max(candidates, key=lambda x: x[1])

    # Convert to tokens, drop SOS
    toks = tgt_vocab.decode(best_tokens[1:])
    # Stop at EOS if present
    if EOS in toks:
        toks = toks[: toks.index(EOS)]
    return ' '.join(toks)

In [ ]:
# Quick qualitative checks
samples = [
    "i like apples.",
    "where is the train station?",
    "can i get some cake?",
    "i am really hungry."
]

for s in samples:
    try:
        print("GD: ", s, "->", greedy_decode(model, s))
        print("TK: ", s, "->", topk_decode(model, s, k=3, temperature=0.8))
        print("TP: ", s, "->", topp_decode(model, s, p=0.5, temperature=0.8))
        print("BS: ", s, "->", beam_search_decode(model, s, beam_width=2))

    except Exception as e:
        print(s, "-> (decoding error)", e)


C:\Users\alombard.PCP-GI-05\miniconda3\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
C:\Users\alombard.PCP-GI-05\AppData\Local\Temp\ipykernel_18632\1591419528.py:18: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  topk_probabilities = torch.nn.functional.softmax(topk.values / temperature)


GD:  i like apples. -> j'aime les pommes .
TK:  i like apples. -> j'aime les pommes .
TP:  i like apples. -> j'aime les pommes .
BS:  i like apples. -> j'aime les pommes .
GD:  where is the train station? -> où est le train de la gare ?
TK:  where is the train station? -> où est le train de la gare ?
TP:  where is the train station? -> où est le train de la gare ?
BS:  where is the train station? -> où est le train de la gare ?
GD:  can i get some cake? -> puis-je me chercher un peu de gâteau ?
TK:  can i get some cake? -> puis-je me chercher un peu de gâteau ?
TP:  can i get some cake? -> puis-je me chercher un peu de gâteau ?
BS:  can i get some cake? -> puis-je avoir un peu de gâteau ?
GD:  i am really hungry. -> j'ai vraiment faim .
TK:  i am really hungry. -> j'ai vraiment faim .
TP:  i am really hungry. -> j'ai vraiment faim .
BS:  i am really hungry. -> j'ai vraiment faim .


In [ ]:
%%sql


In [ ]:
# Show a few random validation samples
for idx in range(min(10, len(val_ds))):
    src, tgt = val_ds[idx]
    src_text = ' '.join(src)
    tgt_text = ' '.join(tgt)
    pred = greedy_decode(model, src_text)
    print(f"EN: {src_text}\nFR(gt): {tgt_text}\nFR(pr): {pred}\n---")

C:\Users\alombard.PCP-GI-05\miniconda3\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


EN: hi .
FR(gt): <sos> salut ! <eos>
FR(pr): choisis .
---
EN: run !
FR(gt): <sos> cours ! <eos>
FR(pr): cours !
---
EN: run !
FR(gt): <sos> courez ! <eos>
FR(pr): cours !
---
EN: who ?
FR(gt): <sos> qui ? <eos>
FR(pr): qui a ?
---
EN: wow !
FR(gt): <sos> ça alors ! <eos>
FR(pr): arrête !
---
EN: fire !
FR(gt): <sos> au feu ! <eos>
FR(pr): bonne feu !
---
EN: help !
FR(gt): <sos> à l'aide ! <eos>
FR(pr): l'aide !
---
EN: jump .
FR(gt): <sos> saute . <eos>
FR(pr): sauter !
---
EN: stop !
FR(gt): <sos> ça suffit ! <eos>
FR(pr): arrête de hurler !
---
EN: stop !
FR(gt): <sos> stop ! <eos>
FR(pr): arrête de hurler !
---
